In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

### PREPROCESAMIENTO

#### 1.- LIMPIEZA

In [ ]:
def limpiar_dataset(df, target_col='Status'):
    
    df_clean = df.dropna()
    
    df_clean = df_clean.drop_duplicates()
    
    feature_cols = [col for col in df_clean.columns if col != target_col]
    df_clean = df_clean.groupby(feature_cols).filter(
        lambda grupo: grupo[target_col].nunique() == 1
    )
    
    return df_clean

#### 2.- CODIFICACIÓN

In [ ]:
def encoding(df, target_col='Status'):
    df_num = df.copy()
    encoders = {}
    
    for col in df_num.columns:
        if df_num[col].dtype == 'object':
            try:
                df_num[col] = pd.to_numeric(df_num[col])
                continue  
            except ValueError:
                pass  
        
        if df_num[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df_num[col]):
            le = LabelEncoder()
            df_num[col] = le.fit_transform(df_num[col].astype(str))
            encoders[col] = le
        elif df_num[col].dtype == 'bool':
            df_num[col] = df_num[col].astype(int)
    return df_num, encoders

#### 3.- BINARIZACIÓN

In [ ]:
def binarizar_dataset(df):

    df_proc = df.copy()
    metadata = {
        'columnas': list(df.columns),
        'minimos': {},
        'factores': {},
        'bits_por_columna': {},
        'desplazamientos': {}  
    }
    
    # Eliminar negativos y decimales
    for col in df.columns:
        datos = df[col].values
        
        min_val = np.min(datos)
        metadata['minimos'][col] = min_val
        if min_val < 0:
            datos = datos - min_val   
        else:
            datos = datos.copy()
        
        max_decimales = 0
        for val in datos:
            if isinstance(val, float) and not val.is_integer():
                s = f"{val:.10f}".rstrip('0')  
                if '.' in s:
                    dec = len(s.split('.')[1])
                    max_decimales = max(max_decimales, dec)
        if max_decimales > 0:
            factor = 10 ** max_decimales
            datos = datos * factor
            datos = np.round(datos).astype(int)
        else:
            factor = 1
            datos = datos.astype(int)
        metadata['factores'][col] = factor
        df_proc[col] = datos
    
    # Determinar bits por columna
    total_bits = 0
    bits_por_col = {}
    for col in df.columns:
        max_val = df_proc[col].max()
        if max_val == 0:
            bits = 1
        else:
            bits = max_val.bit_length()  
        bits_por_col[col] = bits
        metadata['bits_por_columna'][col] = bits
        total_bits += bits
    
    # transformación binaria de cada valor
    n_filas = len(df_proc)
    X_bin = np.zeros((n_filas, total_bits), dtype=int)
    
    inicio_bit = 0
    for col in df.columns:
        bits = bits_por_col[col]
        valores = df_proc[col].values
        for i, val in enumerate(valores):
            bin_str = format(val, f'0{bits}b')
            for j, bit_char in enumerate(bin_str):
                X_bin[i, inicio_bit + j] = int(bit_char)
        inicio_bit += bits
    
    return X_bin, metadata

In [ ]:
def binarizar_data(df):

    df_proc = df.copy()
    metadata = {
        'columnas': list(df.columns),
        'minimos': {},
        'factores': {},
        'bits_por_columna': {},
        'desplazamientos': {}
    }
    
    bit_matrices = []  
    
    for col in df.columns:
        datos = df[col].values.astype(float)  
        
        # Eliminar negativos 
        min_val = np.min(datos)
        metadata['minimos'][col] = min_val
        if min_val < 0:
            datos = datos - min_val
        
        # Eliminar decimales 
        serie = pd.Series(datos)
        max_decimales = serie.astype(str).str.split('.').str[1].str.len().max()
        if pd.isna(max_decimales):
            max_decimales = 0
        
        if max_decimales > 0:
            factor = 10 ** max_decimales
            datos = datos * factor
            datos = np.round(datos).astype(np.int64)
        else:
            factor = 1
            datos = datos.astype(np.int64)
        metadata['factores'][col] = factor
        df_proc[col] = datos
        
        # Valor máximo y bits necesarios 
        max_val = datos.max()
        if max_val == 0:
            bits = 1
        else:
            bits = max_val.bit_length()
        metadata['bits_por_columna'][col] = bits
        
        # Conversión binaria vectorizada -----
        if bits > 0:
            shifts = np.arange(bits-1, -1, -1)  
            bits_matrix = ((datos[:, np.newaxis] >> shifts) & 1).astype(np.int8)
            bit_matrices.append(bits_matrix)
        else:
            bit_matrices.append(np.empty((len(datos), 0), dtype=np.int8))
    
    # Concatenar salida
    if bit_matrices:
        X_bin = np.hstack(bit_matrices)
    else:
        X_bin = np.empty((len(df), 0), dtype=np.int8)
    
    return X_bin, metadata

### OPERADORES

In [4]:
def operadores(x_val, y_val):
    if y_val == 1 and x_val == 1:
        return 1
    elif y_val == 1 and x_val == 0:
        return -1
    else:
        return 0

### DEFINIR DIMENSIONES A UTILIZAR

In [5]:
def valores_importantes(X, Y):
    x_filas, x_columnas = X.shape
    y_filas, y_columnas  = Y.shape

    valores = {
        'x_fil' : x_filas,
        'x_col' : x_columnas,
        'y_fil' : y_filas,
        'y_col' : y_columnas
    }
    return valores

### GENERAR MATRIZ M

In [6]:
def genera_M(m, n):
    return np.zeros((m,n))

### ENTRENAMIENTO

In [7]:
def entrena(X, Y, m, n, muestras, matriz_M):
        
    for k in range(muestras):
        for j in range(m):
            for i in range(n):
                x_val = X[k, i]
                y_val = Y[k, j]
                matriz_M[j, i] += operadores(x_val, y_val)
    
    matriz_entrenada = pd.DataFrame(matriz_M)
    matriz_entrenada = matriz_entrenada.values
    return matriz_entrenada

### RECUPERACIÓN

In [8]:
def recuperacion(matriz_entrenada, x):
    resultados = matriz_entrenada @ x
    return np.argmax(resultados)
    

### Ejemplo de la clase

In [9]:
xs = pd.DataFrame(
    [[1, 0, 1, 0, 1], 
    [1, 1, 0, 0, 1],
    [1, 0, 1, 1, 0]])

ys = pd.DataFrame(
    [[1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]])

x1 = pd.DataFrame([1, 0, 1, 0, 1])
x2 = pd.DataFrame([1, 1, 0, 0, 1])
x3 = pd.DataFrame([1, 0, 1, 1, 0])
#----------------------------------
x4 = pd.DataFrame([0,1,0,1,1]) 
x5 = pd.DataFrame([0,0,1,0,1])            

X = xs.values
Y = ys.values

valores = valores_importantes(X, Y)
m = valores['y_fil']
n = valores['x_col']
muestras = valores['x_fil']

M = genera_M(m, n)
matriz_M = M.copy()
matriz_entrenada = entrena(X, Y, m, n, muestras, matriz_M)

rec_1 = recuperacion(matriz_entrenada, x1)
rec_2 = recuperacion(matriz_entrenada, x2)
rec_3 = recuperacion(matriz_entrenada, x3)
#-----------------------------------------
rec_4 = recuperacion(matriz_entrenada, x4)
rec_5 = recuperacion(matriz_entrenada, x5)


print('===================== EXPERIMENTO DE LA CLASE =====================')
print('\n--------- PATRONES DE ENTRADA --------- ')
print('>>> X')
print(X)
print('\n>>> Y')
print(Y)

print('\n--------- VALORES IMPORTANTES --------- ')
print(f'm: {m}')
print(f'n: {n}')

print('\n--------- GENERAR MATRIZ M VACÍA --------- ')
print(M)

print('\n--------- MATRIZ ENTRENADA (X1, X2, X3) --------- ')
print(matriz_entrenada)

print('\n--------- RECUPERACIÓN --------- ')
print(f'clase X1: 0 \t clase recuperada: {rec_1}')
print(f'clase X2: 1 \t clase recuperada: {rec_2}')
print(f'clase X3: 2 \t clase recuperada: {rec_3}')

print('\n--------- EXTENSIÓN --------- ')
print(f'clase X4: 0 \t clase recuperada: {rec_4}')
print(f'clase X5: 1 \t clase recuperada: {rec_5}')


# VALIDACIÓN LOO
# entrenar sin x1 y depues ver que sale con x1

===================== EXPERIMENTO DE LA CLASE =====================

--------- PATRONES DE ENTRADA --------- 
>>> X
[[1 0 1 0 1]
 [1 1 0 0 1]
 [1 0 1 1 0]]

>>> Y
[[1 0 0]
 [0 1 0]
 [0 0 1]]

--------- VALORES IMPORTANTES --------- 
m: 3
n: 5

--------- GENERAR MATRIZ M VACÍA --------- 
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

--------- MATRIZ ENTRENADA (X1, X2, X3) --------- 
[[ 1. -1.  1. -1.  1.]
 [ 1.  1. -1. -1.  1.]
 [ 1. -1.  1.  1. -1.]]

--------- RECUPERACIÓN --------- 
clase X1: 0 	 clase recuperada: 0
clase X2: 1 	 clase recuperada: 1
clase X3: 2 	 clase recuperada: 2

--------- EXTENSIÓN --------- 
clase X4: 0 	 clase recuperada: 1
clase X5: 1 	 clase recuperada: 0
